# OptiVision measurement batch f8_geometry: vector geometry of models where plain int4 collapses, and of controls (encode only)

Measures 15 models one after the other with the package's own loaders, then zips
the results. Each model is checked on 12 pages first (`check_retriever`); a model that fails
the check or a step is recorded and the batch goes on.

- `lightonai/LateOn`
- `lightonai/mLateOn`
- `mixedbread-ai/mxbai-edge-colbert-v0-17m`
- `mixedbread-ai/mxbai-edge-colbert-v0-32m`
- `lightonai/GTE-ModernColBERT-v1`
- `LiquidAI/LFM2-ColBERT-350M`
- `lightonai/Reason-ModernColBERT`
- `lightonai/ColBERT-Zero`
- `mixedbread-ai/mxbai-colbert-large-v1`
- `colbert-ir/colbertv2.0`
- `nvidia/nemotron-colembed-vl-4b-v2`
- `nvidia/nemotron-colembed-vl-8b-v2`
- `tencent/EVIE-4.5B`
- `tencent/EVIE-8B`
- `vidore/colpali-v1.3`

**Run it:** *Settings -> Accelerator*: **GPU T4 x2**, *Internet*: **On**. Then
*Save Version -> Save & Run All (Commit)* and close the tab; it runs on its own for up to 12 h.
Download `results_f8_geometry.zip` from the version's *Output* tab and give it to Claude.

In [ ]:
BRANCH = "kaggle-campaign"
BATCH = [{'model': 'lightonai/LateOn', 'kind': 'text', 'studies': False, 'limit': 1000},
 {'model': 'lightonai/mLateOn', 'kind': 'text', 'studies': False, 'limit': 1000},
 {'model': 'mixedbread-ai/mxbai-edge-colbert-v0-17m',
  'kind': 'text',
  'studies': False,
  'limit': 1000},
 {'model': 'mixedbread-ai/mxbai-edge-colbert-v0-32m',
  'kind': 'text',
  'studies': False,
  'limit': 1000},
 {'model': 'lightonai/GTE-ModernColBERT-v1', 'kind': 'text', 'studies': False, 'limit': 1000},
 {'model': 'LiquidAI/LFM2-ColBERT-350M', 'kind': 'text', 'studies': False, 'limit': 1000},
 {'model': 'lightonai/Reason-ModernColBERT', 'kind': 'text', 'studies': False, 'limit': 1000},
 {'model': 'lightonai/ColBERT-Zero', 'kind': 'text', 'studies': False, 'limit': 1000},
 {'model': 'mixedbread-ai/mxbai-colbert-large-v1', 'kind': 'text', 'studies': False, 'limit': 1000},
 {'model': 'colbert-ir/colbertv2.0', 'kind': 'text', 'studies': False, 'limit': 1000},
 {'model': 'nvidia/nemotron-colembed-vl-4b-v2',
  'dtype': 'float32',
  'multi_gpu': True,
  'datasets': ['vidore/docvqa_test_subsampled'],
  'studies': False,
  'limit': 100},
 {'model': 'nvidia/nemotron-colembed-vl-8b-v2',
  'dtype': 'float16',
  'multi_gpu': True,
  'datasets': ['vidore/docvqa_test_subsampled'],
  'studies': False,
  'limit': 100},
 {'model': 'tencent/EVIE-4.5B',
  'dtype': 'float16',
  'batch_size': 2,
  'datasets': ['vidore/docvqa_test_subsampled'],
  'studies': False,
  'limit': 100},
 {'model': 'tencent/EVIE-8B',
  'dtype': 'float16',
  'batch_size': 1,
  'multi_gpu': True,
  'datasets': ['vidore/docvqa_test_subsampled'],
  'studies': False,
  'limit': 100,
  'processor_kwargs': {'size': {'longest_edge': 1048576, 'shortest_edge': 65536}}},
 {'model': 'vidore/colpali-v1.3',
  'dtype': 'float32',
  'multi_gpu': True,
  'datasets': ['vidore/docvqa_test_subsampled'],
  'studies': False,
  'limit': 100}]
EXTRA_PIP = ['sentence-transformers[image]>=6.0.0', 'einops']
DEADLINE_HOURS = 11.0
SELECTION = True   # also run the selection study for a model when the session has time to spare

In [ ]:
import json, os, shutil, subprocess, sys, time
if shutil.which("nvidia-smi") is None:
    raise SystemExit("NO GPU IN THIS SESSION. Open the notebook editor, Settings (right panel) -> Accelerator -> "
                     "GPU T4 x2, then Save Version -> Save & Run All (Commit) again. If GPU is greyed out, the "
                     "account needs phone verification (kaggle.com -> Settings -> Phone verification).")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
WORK = "/kaggle/working"
REPO = f"{WORK}/optivision-rag"
NAME = "f8_geometry"
RESULTS = f"{WORK}/results_{NAME}"
os.environ["HF_HOME"] = "/tmp/hf"
!rm -rf {REPO} && git clone -q --depth 1 --branch {BRANCH} https://github.com/Daemon-VI/optivision-rag.git {REPO}
!cd {REPO} && git log --oneline -1
!pip install -q -e "{REPO}[bench,merge,vlm]" 2>&1 | tail -2
!pip install -q -U colpali-engine 2>&1 | tail -1
for package in EXTRA_PIP:
    !pip install -q "{package}" 2>&1 | tail -1
# Kaggle preinstalls torchao 0.10; peft refuses to work while an incompatible torchao is importable
!pip uninstall -y -q torchao
os.makedirs(RESULTS, exist_ok=True)
json.dump(BATCH, open(f"{WORK}/batch.json", "w"), indent=1)

In [ ]:
cmd = [sys.executable, f"{REPO}/scripts/measure_models.py", f"{WORK}/batch.json", "--out", RESULTS,
       "--data", "/tmp/optdata", "--deadline-hours", str(DEADLINE_HOURS)] + (["--selection"] if SELECTION else [])
subprocess.run(cmd, check=False)

In [ ]:
!pip freeze > {RESULTS}/pip_freeze.txt
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv > {RESULTS}/gpu.txt
!cd {REPO} && git rev-parse HEAD > {RESULTS}/commit.txt
!cat {RESULTS}/batch_summary.json
!cd {WORK} && rm -f results_{NAME}.zip && zip -q -r results_{NAME}.zip results_{NAME} && ls -la results_{NAME}.zip